In [109]:
#import of libraries 
import pandas as pd
import numpy as  np 
import seaborn as sbn
import matplotlib.pyplot as plt
import sqlite3


In [110]:
#Ingestion layer
#loading of the data 
df = pd.read_csv("Crime_Data_from_2020_to_2024.csv")

#checking data info 
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1004894 entries, 0 to 1004893
Data columns (total 28 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   DR_NO           1004894 non-null  int64  
 1   Date Rptd       1004894 non-null  object 
 2   DATE OCC        1004894 non-null  object 
 3   TIME OCC        1004894 non-null  int64  
 4   AREA            1004894 non-null  int64  
 5   AREA NAME       1004894 non-null  object 
 6   Rpt Dist No     1004894 non-null  int64  
 7   Part 1-2        1004894 non-null  int64  
 8   Crm Cd          1004894 non-null  int64  
 9   Crm Cd Desc     1004894 non-null  object 
 10  Mocodes         853296 non-null   object 
 11  Vict Age        1004894 non-null  int64  
 12  Vict Sex        860263 non-null   object 
 13  Vict Descent    860251 non-null   object 
 14  Premis Cd       1004878 non-null  float64
 15  Premis Desc     1004306 non-null  object 
 16  Weapon Used Cd  327216 non-null   fl

In [111]:
#understanding the data 
print(df.head())

       DR_NO               Date Rptd                DATE OCC  TIME OCC  AREA  \
0  211507896  04/11/2021 12:00:00 AM  11/07/2020 12:00:00 AM       845    15   
1  201516622  10/21/2020 12:00:00 AM  10/18/2020 12:00:00 AM      1845    15   
2  240913563  12/10/2024 12:00:00 AM  10/30/2020 12:00:00 AM      1240     9   
3  210704711  12/24/2020 12:00:00 AM  12/24/2020 12:00:00 AM      1310     7   
4  201418201  10/03/2020 12:00:00 AM  09/29/2020 12:00:00 AM      1830    14   

     AREA NAME  Rpt Dist No  Part 1-2  Crm Cd  \
0  N Hollywood         1502         2     354   
1  N Hollywood         1521         1     230   
2     Van Nuys          933         2     354   
3     Wilshire          782         1     331   
4      Pacific         1454         1     420   

                                         Crm Cd Desc  ... Status  Status Desc  \
0                                  THEFT OF IDENTITY  ...     IC  Invest Cont   
1     ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT  ...     

In [112]:
#dropping less important columns 
df.drop(columns=["Crm Cd 1",
                    "Crm Cd 2",
                    "Crm Cd 3",
                    "Crm Cd 4",
                    "Cross Street",
                    "LAT",
                    "LON",
                    "DR_NO",
                    "Rpt Dist No" ,
                    "Part 1-2" ,               
                    "Crm Cd",
                    "Status",
                    "Premis Cd",
                    "Vict Descent",
                    "Weapon Used Cd",
                    "LOCATION"],
                    inplace=True)

In [113]:
# checking duplicates
print(df.duplicated().sum())
#dropping the duplicate
df.drop_duplicates(inplace=True)


4875


In [114]:
#checking the missing values 
print(df.isnull().sum())
#dropping the missing value because the impact to the result from data is equal to none
df.dropna(inplace=True)


Date Rptd           0
DATE OCC            0
TIME OCC            0
AREA                0
AREA NAME           0
Crm Cd Desc         0
Mocodes        148726
Vict Age            0
Vict Sex       141801
Premis Desc       587
Weapon Desc    673906
Status Desc         0
dtype: int64


In [115]:
#converting time proper
df["TIME OCC"] = df["TIME OCC"].astype(str).str.zfill(4)
df["hour"] = df["TIME OCC"].str[:2].astype(int)
def time_of_day(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"

df["time_category"] = df["hour"].apply(time_of_day)

# Remove ' 12:00:00 AM' from the date column
df["DATE OCC"]=df["DATE OCC"].str.replace(' 12:00:00 AM', '', regex=False)
df["Date Rptd"]=df["Date Rptd"].str.replace("12:00:00 AM ","",regex=False)
#converting dates 
df["DATE OCC"]=pd.to_datetime(df["DATE OCC"])
df["DATE OCC"]=pd.to_datetime(df["DATE OCC"])
df["Date Rptd"]=pd.to_datetime(df["Date Rptd"])

/tmp/ipykernel_8816/1769177776.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date Rptd"]=pd.to_datetime(df["Date Rptd"])


In [116]:
#cleaning the location columns 
#area ,area_name ,premise desc 
df["AREA"]=pd.to_numeric(df["AREA"],errors ="coerce")
df["AREA NAME"]=df["AREA NAME"].str.strip().str.lower()
df["Premis Desc"]=df["Premis Desc"].str.strip().str.lower()
def categorize_location(x):
    if "dwelling" in x or "apartment" in x:
        return "residential"
    elif "store" in x or "restaurant" in x:
        return "commercial"
    elif "street" in x or "parking" in x:
        return "public"
    else:
        return "other"


df["location_category"] = df["Premis Desc"].apply(categorize_location)

In [117]:
#handling mocode
df["Mocodes"] = df["Mocodes"].str.split(" ")
df = df.explode("Mocodes")

In [118]:
#data storage
df.to_csv("cleaned_lapd_crime.csv")
conn = sqlite3.connect("crime.db")
df.to_sql("crimes", conn, if_exists="replace", index=False)

1555910

In [ ]:
#serving
